# Notebook 2 — Feature Engineering

**Goal:** Transform raw columns into model-ready features.

Steps:
1. Create new features (tenure_group, charges_ratio, num_services, is_month_to_month)
2. Encode categoricals
3. Handle class imbalance with SMOTE
4. Visualise the engineered features

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.feature_engineering import create_features, encode_categoricals, preprocess

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

df = load_data('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('Raw shape:', df.shape)

## 1. Create New Features

In [ ]:
df_feat = create_features(df)

print('New columns added:')
new_cols = ['tenure_group', 'charges_ratio', 'num_services', 'is_month_to_month']
print(df_feat[new_cols].head(10))

## 2. Visualise Engineered Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# tenure_group vs churn
churn_by_tenure = df_feat.groupby('tenure_group')['Churn'].mean() * 100
axes[0].bar(churn_by_tenure.index.astype(str), churn_by_tenure.values,
            color=['#DD8452', '#4C72B0', '#55A868'])
axes[0].set_title('Churn Rate by Tenure Group')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 60)

# num_services vs churn
churn_by_svc = df_feat.groupby('num_services')['Churn'].mean() * 100
axes[1].plot(churn_by_svc.index, churn_by_svc.values, 'o-', color='#4C72B0', lw=2)
axes[1].set_title('Churn Rate by Number of Services')
axes[1].set_xlabel('Number of Active Services')
axes[1].set_ylabel('Churn Rate (%)')

# charges_ratio distribution
for churn_val, label, color in [(0, 'No Churn', '#4C72B0'), (1, 'Churn', '#DD8452')]:
    subset = df_feat[df_feat['Churn'] == churn_val]['charges_ratio']
    axes[2].hist(subset, bins=40, alpha=0.6, label=label, color=color, density=True)
axes[2].set_title('Charges Ratio Distribution')
axes[2].set_xlabel('Monthly / Total Charges Ratio')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. SMOTE — Handle Class Imbalance

In [ ]:
X_train, X_test, y_train, y_test, scaler, feature_names = preprocess(
    df, test_size=0.2, apply_smote=True
)

print(f'\nTrain set shape : {X_train.shape}')
print(f'Test set shape  : {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%}  (balanced after SMOTE)')
print(f'Test churn rate : {y_test.mean():.1%}   (real distribution, untouched)')

print(f'\nFeatures ({len(feature_names)}):')
for f in feature_names:
    print(f'  - {f}')

## Interview note: Why SMOTE only on training data?

SMOTE synthetically creates minority-class samples. If applied before the split, synthetic samples would leak into the test set — the model would be evaluated on artificially generated data, not real customers. Always fit SMOTE on training data only, and evaluate on the original test distribution.